In [ ]:
# =====================================================================
# Does Quantization Hurt Arabic More?  — FINAL zero-edit Kaggle notebook
# Just: GPU T4 x2 + Internet ON -> Save & Run All.   (No code edits needed.)
# Optional: add the previous run's Output as Input -> finished models are skipped (saves ~4h).
#
# Models : Qwen3-4B, Llama-3.2-3B-Instruct + one third family chosen automatically:
#          first of THIRD_CANDIDATES whose files exist AND passes the sanity checks.
# Levels : F16 baseline, static Q8_0..Q2_K, imatrix Q6_K..Q2_K (same quantizer: mradermacher)
# Safety : tokenizer check + F16 accuracy check -> a broken model is skipped automatically
#          and excluded from the analysis; every finished run is saved immediately.
# OUTPUT (/kaggle/working): results.csv, samples.csv, fertility.csv, discovery.csv,
#          analysis_rq1.csv, analysis_rq2.csv
# =====================================================================
import subprocess, sys, os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
pip = lambda *a: subprocess.run([sys.executable, "-m", "pip", "-q", "install", *a], check=True)
pip("git+https://github.com/EleutherAI/lm-evaluation-harness")
pip("transformers>=4.57,<4.58", "gguf", "huggingface_hub", "accelerate")

import re, gc, glob, shutil, traceback, numpy as np, pandas as pd, torch
from huggingface_hub import list_repo_files, hf_hub_download

# ------------------------- CONFIG (no edits needed) -------------------------
WORK, CACHE = "/kaggle/working", "/tmp/models"
CORE   = {"Qwen3-4B": "Qwen3-4B", "Llama-3.2-3B-Instruct": "Llama-3.2-3B-Instruct"}
THIRD_CANDIDATES = {"Phi-3.5-mini-instruct": "Phi-3.5-mini-instruct", "Qwen2.5-3B-Instruct": "Qwen2.5-3B-Instruct"}
MODELS = {**CORE, **THIRD_CANDIDATES}
F16_FALLBACK = {"Qwen3-4B": ["bartowski/Qwen_Qwen3-4B-GGUF"], "Llama-3.2-3B-Instruct": ["bartowski/Llama-3.2-3B-Instruct-GGUF"],
                "Phi-3.5-mini-instruct": ["bartowski/Phi-3.5-mini-instruct-GGUF"],
                "Qwen2.5-3B-Instruct": ["bartowski/Qwen2.5-3B-Instruct-GGUF", "Qwen/Qwen2.5-3B-Instruct-GGUF"]}
STATIC_Q  = ["Q8_0", "Q6_K", "Q4_K_M", "Q3_K_M", "Q2_K"]
IMATRIX_Q = ["Q6_K", "Q4_K_M", "Q3_K_M", "Q2_K"]
TASKS = ["global_mmlu_ar", "global_mmlu_en", "belebele_arb_Arab", "belebele_eng_Latn"]
BS, LIMIT, N_BOOT, CHANCE = 8, None, 10000, 0.25
MIN_F16_ACC, MAX_EN_TPW = 0.30, 1.8
tag = ""
P = {k: f"{WORK}/{k}.csv" for k in ["results", "samples", "fertility", "discovery", "analysis_rq1", "analysis_rq2"]}
os.makedirs(WORK, exist_ok=True)

def load_all(name):  # previous runs (Input) first, this run last -> this run wins on duplicates
    paths = [p for p in sorted(glob.glob(f"/kaggle/input/**/{name}.csv", recursive=True)) + [P[name]] if os.path.exists(p)]
    if not paths: raise FileNotFoundError(f"{name}.csv not found")
    return pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)

def broken_models(res):  # any model whose F16 baseline is near chance is excluded everywhere
    f = res[res.quant == "F16"].groupby("model").acc.min()
    return set(f[f < MIN_F16_ACC].index)

# ------------------------- FILE DISCOVERY -------------------------
_cache = {}
def repo_files(repo):
    if repo not in _cache:
        try: _cache[repo] = [f for f in list_repo_files(repo) if f.endswith(".gguf") and "mmproj" not in f.lower() and "-of-" not in f]
        except Exception: _cache[repo] = []
    return _cache[repo]

def locate(model, variant, q):
    base = MODELS[model]
    repos = [f"mradermacher/{base}-i1-GGUF"] if variant == "imatrix" else \
            [f"mradermacher/{base}-GGUF"] + (F16_FALLBACK[model] if q == "F16" else [])
    pats = [r"(^|[-_.])f16\.gguf$", r"(^|[-_.])fp16\.gguf$", r"(^|[-_.])bf16\.gguf$"] if q == "F16" else [rf"(^|[-_.]){q}\.gguf$"]
    for repo in repos:
        for p in pats:
            hit = [f for f in repo_files(repo) if re.search(p, os.path.basename(f), re.I)
                   and (q == "F16" or (".i1-" in os.path.basename(f)) == (variant == "imatrix"))]
            if hit: return repo, sorted(hit, key=len)[0]
    return None, None

plan_of = lambda m: [(m, "static", "F16")] + [(m, "static", q) for q in STATIC_Q] + [(m, "imatrix", q) for q in IMATRIX_Q]
disc = pd.DataFrame([dict(model=m, variant=v, quant=q, **dict(zip(["repo", "file"], locate(m, v, q))))
                     for m in MODELS for (_, v, q) in plan_of(m)])
disc.to_csv(P["discovery"], index=False)
complete = {m for m, g in disc.groupby("model") if g.file.notna().all()}
print("FILE DISCOVERY:\n", disc.to_string(index=False), "\nComplete models:", sorted(complete), flush=True)

# ------------------------- HELPERS -------------------------
lang_of  = lambda t: "ar" if ("_ar" in t or "arb_" in t) else "en"
bench_of = lambda t: "GlobalMMLU" if t.startswith("global_mmlu") else "Belebele"
def item_key(doc, doc_id):
    if "sample_id" in doc: return str(doc["sample_id"])
    if "link" in doc and "question_number" in doc: return f'{doc["link"]}|{doc["question_number"]}'
    return str(doc_id)
def append(path, rows):
    if rows: pd.DataFrame(rows).to_csv(path, mode="a", header=not os.path.exists(path), index=False)

# ------------------------- EVALUATION -------------------------
import lm_eval
from lm_eval.models.huggingface import HFLM

try: prev = load_all("results")
except FileNotFoundError: prev = pd.DataFrame(columns=["model", "variant", "quant", "acc"])
bad = broken_models(prev) if len(prev) else set()
done = {tuple(r) for r in prev[~prev.model.isin(bad)][["model", "variant", "quant"]].drop_duplicates().values}
third = next((m for m in THIRD_CANDIDATES if (m, "static", "F16") in done), None)   # already chosen in an earlier run
print(f"Already done: {len(done)} runs | excluded as broken: {sorted(bad)} | third model so far: {third}", flush=True)

def evaluate(m, v, q):
    """Returns 'ok', 'broken' or 'failed'."""
    repo, fname = locate(m, v, q)
    print(f"\n=== {m} | {v} | {q} | {repo}/{fname}", flush=True)
    local = f"{CACHE}/{m}"; lm = out = None
    try:
        hf_hub_download(repo, fname, local_dir=local)
        size_gb = os.path.getsize(f"{local}/{fname}") / 1e9
        lm = HFLM(pretrained=local, gguf_file=fname, dtype="float16", batch_size=BS, device="cuda:0")
        sent = "The quick brown fox jumps over the lazy dog near the old river bank today."
        tpw = len(lm.tokenizer(sent, add_special_tokens=False).input_ids) / len(sent.split())
        print(f"tokenizer check: {tpw:.2f} English tokens/word", flush=True)
        if tpw > MAX_EN_TPW:
            print(f"!! SANITY FAIL {m}: tokenizer broken -> model skipped", flush=True); return "broken"
        out = lm_eval.simple_evaluate(model=lm, tasks=TASKS, num_fewshot=0, log_samples=True, limit=LIMIT)
        allres = {**out.get("results", {}), **out.get("groups", {})}
        accs = {t: allres[t]["acc,none"] for t in TASKS}
        print(pd.Series(accs).round(4).to_string(), flush=True)
        if q == "F16" and min(accs.values()) < MIN_F16_ACC:
            print(f"!! SANITY FAIL {m}: F16 accuracy near chance -> model skipped", flush=True); return "broken"
        append(P["results"], [dict(model=m, variant=v, quant=q, repo=repo, file=fname, size_gb=round(size_gb, 3),
                                   task=t, bench=bench_of(t), lang=lang_of(t), acc=a) for t, a in accs.items()])
        append(P["samples"], [dict(model=m, variant=v, quant=q, bench=bench_of(st), lang=lang_of(st),
                                   key=item_key(s["doc"], s["doc_id"]), correct=int(s["acc"]))
                              for st, lst in out["samples"].items() for s in lst])
        if q == "F16":
            fr = []
            for st, lst in out["samples"].items():
                if st.startswith("belebele"):
                    ps = list({s["doc"]["flores_passage"] for s in lst})
                    ntok = sum(len(lm.tokenizer(p, add_special_tokens=False).input_ids) for p in ps)
                    fr.append(dict(model=m, lang=lang_of(st), passages=len(ps), tokens_per_word=ntok / sum(len(p.split()) for p in ps)))
            append(P["fertility"], fr)
        return "ok"
    except Exception:
        print(f"!! FAILED {m} {v} {q}\n{traceback.format_exc()[-1500:]}", flush=True); return "failed"
    finally:
        del lm, out; gc.collect(); torch.cuda.empty_cache(); shutil.rmtree(local, ignore_errors=True)

for m in list(CORE) + list(THIRD_CANDIDATES):
    if m in THIRD_CANDIDATES and third and m != third: continue          # a third model is already chosen
    if m not in complete: print(f"skip {m}: files incomplete"); continue
    f16_ok = (m, "static", "F16") in done
    for (_, v, q) in plan_of(m):
        if (m, v, q) in done: continue
        status = evaluate(m, v, q)
        if q == "F16":
            f16_ok = status == "ok"
            if not f16_ok: break                                         # no valid baseline -> drop model
    if m in THIRD_CANDIDATES and f16_ok: third = m
    print(f"--> {m}: {'kept' if f16_ok else 'dropped'}", flush=True)
print(f"\nThird model used: {third}", flush=True)

# ------------------------- STATISTICS -------------------------
# Two metrics, both reported:
#  raw  : accuracy-point drop (F16 -> quant)
#  norm : share of above-chance knowledge lost = drop / (acc_F16 - 0.25)  -> removes floor effect
# Paired item-level bootstrap (items resampled jointly across ar/en), two-sided bootstrap p,
# Holm correction within each family of tests.
rng = np.random.default_rng(0)
def boot_p(b):
    return float(min(1.0, 2 * min((b <= 0).mean(), (b >= 0).mean())))
def holm(p):
    p = np.asarray(p, float); adj = np.empty_like(p); run = 0.0
    for r, i in enumerate(np.argsort(p)): run = max(run, (len(p) - r) * p[i]); adj[i] = min(1.0, run)
    return adj
def ci(b): return np.percentile(b, 2.5), np.percentile(b, 97.5)

s = load_all("samples").drop_duplicates(["model", "variant", "quant", "bench", "lang", "key"], keep="last")
_bad = broken_models(load_all("results")); s = s[~s.model.isin(_bad)]
print("Models in analysis:", sorted(s.model.unique()), "| excluded:", sorted(_bad))
def col(m, v, q, b, l):
    return s[(s.model == m) & (s.variant == v) & (s.quant == q) & (s.bench == b) & (s.lang == l)].set_index("key").correct.rename(f"{v}|{q}|{l}")

def diff_stats(A0, A1, E0, E1, H_A, H_E):
    """A0/A1, E0/E1: reference/compare correctness arrays (ar/en). H_*: headroom arrays for normalisation.
    Returns point + bootstrap distributions for raw and normalised (ar - en) differences of (A0-A1) vs (E0-E1)."""
    n = len(A0); idx = rng.integers(0, n, (N_BOOT, n))
    def f(i):
        da, de = A0[i].mean(-1) - A1[i].mean(-1), E0[i].mean(-1) - E1[i].mean(-1)
        ha, he = np.clip(H_A[i].mean(-1) - CHANCE, 1e-6, None), np.clip(H_E[i].mean(-1) - CHANCE, 1e-6, None)
        return da * 100, de * 100, (da - de) * 100, da / ha * 100, de / he * 100, (da / ha - de / he) * 100
    return f(np.arange(n)), f(idx)

# ---- RQ1: does quantization hurt Arabic more? (reference = F16)
rq1 = []
for (m, v, q, b), _ in s[s.quant != "F16"].groupby(["model", "variant", "quant", "bench"]):
    d = pd.concat([col(m, "static", "F16", b, "ar"), col(m, v, q, b, "ar"), col(m, "static", "F16", b, "en"), col(m, v, q, b, "en")], axis=1).dropna()
    if d.empty: continue
    a0, a1, e0, e1 = d.values.T
    pt, bs = diff_stats(a0, a1, e0, e1, a0, e0)
    (lr, hr), (ln, hn) = ci(bs[2]), ci(bs[5])
    rq1.append(dict(model=m, variant=v, quant=q, bench=b, n_items=len(d), accF16_ar=a0.mean() * 100, accF16_en=e0.mean() * 100,
                    drop_ar=pt[0], drop_en=pt[1], excess_raw=pt[2], raw_lo=lr, raw_hi=hr, p_raw=boot_p(bs[2]),
                    lost_ar_pct=pt[3], lost_en_pct=pt[4], excess_norm=pt[5], norm_lo=ln, norm_hi=hn, p_norm=boot_p(bs[5])))
rq1 = pd.DataFrame(rq1)
rq1["p_raw_holm"], rq1["p_norm_holm"] = holm(rq1.p_raw), holm(rq1.p_norm)
rq1.to_csv(P["analysis_rq1"], index=False)

# ---- RQ2: does imatrix help Arabic as much as English? (reference = static, same quant; normalised by F16 headroom)
rq2 = []
for (m, q, b), _ in s[s.variant == "imatrix"].groupby(["model", "quant", "bench"]):
    d = pd.concat([col(m, "imatrix", q, b, "ar"), col(m, "static", q, b, "ar"), col(m, "imatrix", q, b, "en"), col(m, "static", q, b, "en"),
                   col(m, "static", "F16", b, "ar"), col(m, "static", "F16", b, "en")], axis=1).dropna()
    if d.empty: continue
    ia, sa, ie, se, fa, fe = d.values.T
    pt, bs = diff_stats(ia, sa, ie, se, fa, fe)   # gain = imatrix - static
    (lr, hr), (ln, hn) = ci(bs[2]), ci(bs[5])
    rq2.append(dict(model=m, quant=q, bench=b, n_items=len(d), gain_ar=pt[0], gain_en=pt[1], gain_diff_raw=pt[2], raw_lo=lr, raw_hi=hr,
                    p_raw=boot_p(bs[2]), gain_ar_pct=pt[3], gain_en_pct=pt[4], gain_diff_norm=pt[5], norm_lo=ln, norm_hi=hn, p_norm=boot_p(bs[5])))
rq2 = pd.DataFrame(rq2)
if len(rq2): rq2["p_raw_holm"], rq2["p_norm_holm"] = holm(rq2.p_raw), holm(rq2.p_norm)
rq2.to_csv(P["analysis_rq2"], index=False)

# ---- Print
pd.set_option("display.width", 300); pd.set_option("display.max_columns", 50)
qo = {q: i for i, q in enumerate(["Q8_0", "Q6_K", "Q4_K_M", "Q3_K_M", "Q2_K"])}
rq1 = rq1.assign(o=rq1.quant.map(qo)).sort_values(["o", "bench", "model", "variant"]).drop(columns="o")
show1 = ["model", "variant", "quant", "bench", "excess_raw", "p_raw_holm", "lost_ar_pct", "lost_en_pct", "excess_norm", "norm_lo", "norm_hi", "p_norm_holm"]
print("\n=== RQ1: excess > 0 => Arabic hurt MORE (raw = points; norm = % of above-chance knowledge) ===\n", rq1[show1].round(3).to_string(index=False))
print("\nSign consistency (excess_norm > 0) and Holm-significant cells per quant:")
print(rq1.groupby("quant").apply(lambda g: pd.Series(dict(cells=len(g), positive=int((g.excess_norm > 0).sum()),
      sig_norm=int((g.p_norm_holm < .05).sum()), sig_raw=int((g.p_raw_holm < .05).sum())))).reindex([q for q in qo if q in set(rq1.quant)]).to_string())
if len(rq2):
    show2 = ["model", "quant", "bench", "gain_ar", "gain_en", "gain_diff_raw", "p_raw_holm", "gain_ar_pct", "gain_en_pct", "gain_diff_norm", "norm_lo", "norm_hi", "p_norm_holm"]
    print("\n=== RQ2: gain_diff < 0 => imatrix helps Arabic LESS than English ===\n", rq2[show2].round(3).to_string(index=False))
try: print("\n=== Tokenizer fertility ===\n", load_all("fertility").drop_duplicates(["model", "lang"], keep="last").round(3).to_string(index=False))
except FileNotFoundError: pass
